In [ ]:
# -*- coding: utf-8 -*-
"""
Autoencoder físico guiado — VERSÃO ULTRA ESTÁVEL PARA JUPYTERLAB
Compatível com CPU fraca e Windows (sem reiniciar kernel).
Autor: Luiz Eduardo Abdala José
"""

import os, gc, warnings
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["MPLCONFIGDIR"] = os.getcwd() + "/.matplotlib"
warnings.filterwarnings("ignore", category=UserWarning)

import torch, torch.nn as nn, torch.optim as optim
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("svg")  # ✅ backend 100% estável para JupyterLab
import matplotlib.pyplot as plt

torch.set_num_threads(1)
plt.ioff()
os.makedirs("resultados", exist_ok=True)

# ======================================================
# 1. CARREGAR BASE
# ======================================================
PKL_TREINO = "base_treino.pkl"
REF_TEMP = 20

base_tr = pd.read_pickle(PKL_TREINO)
f_cols = [c for c in base_tr.columns if c.startswith("f_")]
freqs = np.array([float(c[2:-2]) for c in f_cols])

X = base_tr[f_cols].to_numpy(float)
T = base_tr["temp_c"].to_numpy(float)

# Subamostragem e normalização
X = X[:, ::10]
freqs = freqs[::10]
X = (X - X.mean(axis=1, keepdims=True)) / (X.std(axis=1, keepdims=True) + 1e-9)
Tn = (T - T.min()) / (T.max() - T.min())

y_ref = np.median(base_tr.loc[base_tr["temp_c"] == REF_TEMP, f_cols].to_numpy(float), axis=0)[::10]
y_ref = (y_ref - y_ref.mean()) / (y_ref.std() + 1e-9)
Y_ref = np.repeat(y_ref[None, :], len(X), axis=0)

X_train = torch.tensor(X, dtype=torch.float32)
Y_ref_t = torch.tensor(Y_ref, dtype=torch.float32)
T_train = torch.tensor(Tn, dtype=torch.float32).unsqueeze(1)

# Limite de dados para segurança
MAX_SAMPLES = 300
if len(X_train) > MAX_SAMPLES:
    X_train, Y_ref_t, T_train, T = X_train[:MAX_SAMPLES], Y_ref_t[:MAX_SAMPLES], T_train[:MAX_SAMPLES], T[:MAX_SAMPLES]

input_dim = X_train.shape[1]
device = "cpu"

# ======================================================
# 2. MODELO
# ======================================================
u_global = torch.linspace(-0.5, 0.5, input_dim).unsqueeze(0)

class PhysicsGuidedAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=12):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim + 1, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, latent_dim)
        )

    def forward(self, x, t):
        u = u_global.to(x.device)
        z = self.encoder(torch.cat([x, t], dim=1))
        (offset, gain, slope, shift,
         skew, kurt, ent, amp,
         centroid, rough, std_adj, bias) = torch.split(z, 1, dim=1)

        x_c = x + offset
        x_mean = x_c.mean(dim=1, keepdims=True)
        x_c = x_mean + gain*(x_c - x_mean)
        x_c = x_c + slope*u
        x_c = x_c + shift*torch.gradient(x_c, dim=1)[0]*0.05
        x_c = x_c + skew*u**2 + kurt*u**4
        x_c = x_c + ent*torch.sin(np.pi*u)
        x_c = x_c + rough*torch.cos(2*np.pi*u)
        x_c = x_c + centroid*u
        x_c = x_c*(1 + amp) + std_adj*0.01 + bias*0.05
        x_c = (x_c - x_c.mean(dim=1, keepdims=True)) / (x_c.std(dim=1, keepdims=True) + 1e-9)
        return x_c, z

# ======================================================
# 3. TREINAMENTO
# ======================================================
model = PhysicsGuidedAutoencoder(input_dim=input_dim).to(device)
opt = optim.Adam(model.parameters(), lr=5e-4)
loss_fn = nn.MSELoss()

epochs = 60
batch_size = 16
loss_hist = []

print("Treinando modelo leve e estável (CPU)...\n")
with torch.autograd.set_detect_anomaly(False):
    for epoch in range(epochs):
        perm = torch.randperm(X_train.size(0))
        total_loss = 0.0
        for i in range(0, len(perm), batch_size):
            idx = perm[i:i+batch_size]
            xb, tb, yb = X_train[idx], T_train[idx], Y_ref_t[idx]
            recon, z = model(xb, tb)
            loss = loss_fn(recon, yb) + 1e-4*torch.mean(z**2)
            opt.zero_grad(); loss.backward(); opt.step()
            total_loss += loss.item()*len(idx)
            del xb, tb, yb, recon, z
        loss_hist.append(total_loss/len(X_train))
        if epoch % 10 == 0:
            print(f"Época {epoch:03d} | Loss = {loss_hist[-1]:.6f}")
        gc.collect()

print("\n✅ Treinamento concluído com sucesso!\n")

# ======================================================
# 4. PLOTS SIMPLIFICADOS
# ======================================================
def salvar_figura_segura(fig, nome):
    fig.savefig(f"resultados/{nome}", dpi=200)
    plt.close(fig)
    gc.collect()
    print(f"Figura salva: {nome}")

# Loss simples
fig = plt.figure()
plt.plot(loss_hist)
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Perda durante o treino")
plt.grid(True)
salvar_figura_segura(fig, "01_loss.svg")

# Parâmetro vs T (só 1 parâmetro)
with torch.no_grad():
    z_all = []
    for i in range(0, len(X_train), batch_size):
        z_all.append(model(X_train[i:i+batch_size], T_train[i:i+batch_size])[1])
    z = torch.cat(z_all, dim=0).cpu().numpy()

fig = plt.figure()
plt.plot(T, z[:, 0], ".", alpha=0.7)
plt.xlabel("Temperatura (°C)")
plt.ylabel("Parâmetro 0")
plt.title("Parâmetro físico vs T")
plt.grid(True)
salvar_figura_segura(fig, "02_parametro.svg")

# Reconstrução exemplo (linha única)
idx = np.argmin(np.abs(T - 70))
with torch.no_grad():
    recon, _ = model(X_train[idx:idx+1], T_train[idx:idx+1])
recon_np = recon.cpu().numpy().ravel()

fig = plt.figure()
plt.plot(freqs/1e3, X[idx], label=f"{T[idx]:.0f}°C", color="red", lw=1)
plt.plot(freqs/1e3, recon_np, label="Compensada", color="blue", lw=1)
plt.plot(freqs/1e3, y_ref, "--", label="20°C", color="black", lw=0.8)
plt.xlabel("Frequência (kHz)")
plt.ylabel("Magnitude normalizada")
plt.title("Compensação térmica (simples)")
plt.grid(True)
plt.legend(fontsize=7)
salvar_figura_segura(fig, "03_reconstrucao.svg")

print("✅ Todas as figuras foram salvas em ./resultados/ com segurança.")
print("✅ Kernel 100% estável para JupyterLab 🚀")


In [ ]:
# ======================================================
# GRÁFICO ÚNICO — COMPENSAÇÃO DE TEMPERATURA
# ======================================================
import gc, matplotlib
matplotlib.use("svg")
import matplotlib.pyplot as plt

def salvar_figura_segura(fig, nome):
    fig.savefig(f"resultados/{nome}", dpi=200)
    plt.close(fig)
    gc.collect()
    print(f"✅ Figura salva: resultados/{nome}")

# Seleciona uma temperatura de exemplo (ajuste se quiser)
idx = np.argmin(np.abs(T - 70))

# Reconstrução com o modelo já treinado
with torch.no_grad():
    recon, _ = model(X_train[idx:idx+1], T_train[idx:idx+1])
recon_np = recon.cpu().numpy().ravel()

# Plot simples
fig = plt.figure(figsize=(6,4))
plt.plot(freqs/1e3, X[idx], label=f"Original {T[idx]:.0f}°C", color='tab:red', lw=1)
plt.plot(freqs/1e3, recon_np, label="Compensada", color='tab:blue', lw=1)
plt.plot(freqs/1e3, y_ref, '--', label="Referência 20°C", color='black', lw=0.8)
plt.xlabel("Frequência (kHz)")
plt.ylabel("Magnitude normalizada")
plt.title("Compensação térmica")
plt.grid(True)
plt.legend(fontsize=8)
salvar_figura_segura(fig, "compensacao.svg")


In [ ]:
# -*- coding: utf-8 -*-
"""
Autoencoder — Compensação de Temperatura (versão leve e estável)
Autor: Luiz Eduardo Abdala José
"""

import os, gc, warnings
warnings.filterwarnings("ignore", category=UserWarning)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["MPLCONFIGDIR"] = os.getcwd() + "/.matplotlib"

import torch, torch.nn as nn
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("svg")
import matplotlib.pyplot as plt
plt.ioff()

os.makedirs("resultados", exist_ok=True)

# ======================================================
# 1. CARREGAR BASE
# ======================================================
PKL_TREINO = "base_treino.pkl"
REF_TEMP = 20

base_tr = pd.read_pickle(PKL_TREINO)
f_cols = [c for c in base_tr.columns if c.startswith("f_")]
freqs = np.array([float(c[2:-2]) for c in f_cols])

X = base_tr[f_cols].to_numpy(float)
T = base_tr["temp_c"].to_numpy(float)

# Reduz pontos para leveza
X = X[:, ::10]
freqs = freqs[::10]

# Normalização simples
X = (X - X.mean(axis=1, keepdims=True)) / (X.std(axis=1, keepdims=True) + 1e-9)
Tn = (T - T.min()) / (T.max() - T.min())

# Curva de referência 20°C
y_ref = np.median(base_tr.loc[base_tr["temp_c"] == REF_TEMP, f_cols].to_numpy(float), axis=0)[::10]
y_ref = (y_ref - y_ref.mean()) / (y_ref.std() + 1e-9)

# ======================================================
# 2. MODELO AUTOENCODER LEVE
# ======================================================
class SimpleAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=12):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim + 1, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim + 1, 32), nn.ReLU(),
            nn.Linear(32, 64), nn.ReLU(),
            nn.Linear(64, input_dim)
        )

    def forward(self, x, t):
        z = self.encoder(torch.cat([x, t], dim=1))
        x_hat = self.decoder(torch.cat([z, t], dim=1))
        return x_hat, z

# ======================================================
# 3. TREINAR (POUCAS ÉPOCAS)
# ======================================================
X_t = torch.tensor(X, dtype=torch.float32)
T_t = torch.tensor(Tn, dtype=torch.float32).unsqueeze(1)
Y_ref_t = torch.tensor(y_ref, dtype=torch.float32).unsqueeze(0).repeat(X_t.shape[0], 1)

model = SimpleAutoencoder(X.shape[1])
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(50):  # leve, rápido
    opt.zero_grad()
    recon, _ = model(X_t, T_t)
    loss = loss_fn(recon, Y_ref_t)
    loss.backward()
    opt.step()
    if epoch % 10 == 0:
        print(f"Época {epoch:03d} | Loss = {loss.item():.5f}")

# ======================================================
# 4. GERAR UM EXEMPLO (20°C vs 70°C)
# ======================================================
idx = np.argmin(np.abs(T - 70))
with torch.no_grad():
    recon, _ = model(X_t[idx:idx+1], T_t[idx:idx+1])
recon_np = recon.numpy().ravel()

# ======================================================
# 5. PLOT ÚNICO
# ======================================================
def salvar_figura_segura(fig, nome):
    fig.savefig(f"resultados/{nome}", dpi=200)
    plt.close(fig)
    gc.collect()
    print(f"✅ Figura salva: resultados/{nome}")

fig = plt.figure(figsize=(6,4))
plt.plot(freqs/1e3, X[idx], label=f"Original {T[idx]:.0f}°C", color='tab:red', lw=1)
plt.plot(freqs/1e3, recon_np, label="Compensada (Autoencoder)", color='tab:blue', lw=1)
plt.plot(freqs/1e3, y_ref, '--', label="Referência 20°C", color='black', lw=0.8)
plt.xlabel("Frequência (kHz)")
plt.ylabel("Magnitude normalizada")
plt.title("Compensação térmica — Exemplo único (70°C → 20°C)")
plt.grid(True)
plt.legend(fontsize=8)
salvar_figura_segura(fig, "compensacao_unica.svg")

print("\n✅ Gráfico gerado com sucesso: ./resultados/compensacao_unica.svg")
